# Mattina 1 — Oltre l'accuracy: metriche per la PD e primi passi con PyTorch

**Corso «Deep Learning per il Rischio di Credito: stima della PD con PyTorch»** — notebook di laboratorio della prima mattina.

In questa mattina costruiamo le fondamenta di tutto il corso:

1. sostituiamo l'**accuracy** — oggi l'unica metrica in uso — con il set di metriche che la Vigilanza si aspetta per un modello di PD (**discriminazione** e **calibrazione**);
2. costruiamo le prime **baseline**: il modello banale, la regressione logistica e **XGBoost**, che da qui in avanti sara' il nostro *champion* (il modello da battere);
3. muoviamo i primi passi in **PyTorch** e addestriamo il primo **MLP** (percettrone multistrato), confrontandolo con le baseline sulle stesse metriche.

**Come usare questo notebook in Visual Studio Code**: aprire il file, selezionare in alto a destra il kernel dell'ambiente conda del corso, ed eseguire le celle **in ordine, dall'alto verso il basso** (pulsante ▶ a sinistra di ogni cella, oppure `Maiusc+Invio`). Ogni cella di codice e' preceduta da una spiegazione: leggiamo prima, eseguiamo poi.

## Assunzioni di questo notebook

Esplicitiamo tutte le assunzioni, cosi' nessun risultato va interpretato oltre il suo perimetro.

| # | Assunzione |
|---|-----------|
| 1 | **Dataset**: *Default of Credit Card Clients* (Taiwan, UCI) — 30.000 clienti di carte di credito, storico di 6 mesi. Il file va posizionato in `dati/credit_default_taiwan.csv`. In sua assenza il notebook usa, se presente, il segnaposto sintetico `dati/credit_default_taiwan_sintetico.csv`, che replica lo schema del dataset reale e serve solo a far girare il codice |
| 2 | **Target**: la colonna `default` vale 1 se il cliente e' andato in default **il mese successivo**. E' un orizzonte di 1 mese: lo usiamo come **proxy didattico** della PD a orizzonte fisso. Nel mondo reale il riferimento e' la definizione regolamentare (art. 178 CRR) con orizzonte a 12 mesi: i concetti e il codice restano identici, cambia solo l'etichetta |
| 3 | **Soglie di lettura del Gini** (debole < 40% ≤ accettabile ≤ 60% < buono): sono **prassi di validazione**, non valori imposti da una norma |
| 4 | **Ambiente**: tutto gira su **CPU** in pochi minuti; nessuna GPU richiesta |
| 5 | **Riproducibilita'**: fissiamo un seme (`SEED = 42`) per rendere ripetibili campionamenti e addestramenti |

## Glossario — acronimi e concetti usati in questa mattina

Tutti i termini che incontreremo, spiegati una volta per tutte. Torniamo pure qui ogni volta che serve.

| Termine | Significato |
|---|---|
| **PD** (*probability of default*) | probabilita' che la controparte vada in default entro l'orizzonte scelto |
| **Default** | inadempienza del cliente; nel dataset e' la colonna `default` (0/1) |
| **Classificazione binaria** | prevedere una tra due classi (default si'/no); il modello emette una **probabilita'** |
| **Accuracy** | quota di previsioni corrette sul totale; inganna quando le classi sono sbilanciate |
| **Sbilanciamento** (*class imbalance*) | una classe e' molto piu' rara dell'altra (i default sono pochi) |
| **Matrice di confusione** | tabella 2×2 dei conteggi: veri/falsi positivi e veri/falsi negativi |
| **TP, FP, TN, FN** | *true/false positive*, *true/false negative*: le quattro celle della matrice |
| **Precision** | tra i segnalati come «default», quanti lo sono davvero: TP/(TP+FP) |
| **Recall** (o TPR) | tra i default veri, quanti ne troviamo: TP/(TP+FN) |
| **F1** | media armonica di precision e recall: equilibrio tra le due |
| **Soglia** (*threshold*) | valore sopra il quale la probabilita' diventa «classe 1»; di default 0,5, ma si sceglie |
| **FPR** | *false positive rate*: quota di buoni clienti segnalati per errore: FP/(FP+TN) |
| **Curva ROC** | TPR contro FPR al variare della soglia: fotografia della capacita' di ordinamento |
| **AUC** | area sotto la curva ROC (0,5 = caso; 1 = perfetto) |
| **Gini** | = 2·AUC − 1; stessa informazione dell'AUC, nella scala usata dai validatori |
| **PR-AUC** | area sotto la curva Precision-Recall; piu' severa dell'AUC sulle classi rare |
| **KS** (Kolmogorov–Smirnov) | massima distanza tra le distribuzioni dei punteggi di buoni e cattivi |
| **Discriminazione** | capacita' del modello di *ordinare* i clienti dal meno al piu' rischioso |
| **Calibrazione** | corrispondenza tra probabilita' stimate e frequenze osservate (3% stimato → ~3% osservato) |
| **Reliability diagram** | grafico della calibrazione: PD stimata contro tasso osservato, per gruppi |
| **Brier score** | errore quadratico medio sulle probabilita'; piu' basso = meglio |
| **Fascia di rating** | gruppo di clienti con PD simile; i test di calibrazione si fanno per fascia |
| **Test binomiale** | test statistico: il numero di default osservati in fascia e' compatibile con la PD stimata? |
| **Test di Jeffreys** | variante bayesiana del binomiale (distribuzione Beta), prevista dalle istruzioni BCE |
| **p-value** | probabilita' di osservare dati cosi' estremi se l'ipotesi del test fosse vera; piccolo = allarme |
| **Champion / challenger** | modello in produzione / modello sfidante, confrontati su stessi dati e metriche |
| **Baseline** | modello semplice di riferimento, da battere |
| **Train / validation / test** | dati per addestrare / per le scelte intermedie / per il giudizio finale, mai mescolati |
| **Overfitting** | il modello impara il rumore del campione e peggiora su dati nuovi |
| **Scaling** | riportare le variabili su scale comparabili; serve alle reti, non agli alberi |
| **Tensore** | l'array multidimensionale di PyTorch (il cugino del `ndarray` di NumPy) |
| **Autograd** | il motore di PyTorch che calcola automaticamente le derivate (gradienti) |
| **Gradiente** | direzione di massima crescita della loss; si scende nella direzione opposta |
| **Loss** (funzione di perdita) | misura dell'errore che l'addestramento minimizza |
| **BCE** (*binary cross-entropy*) | la loss standard della classificazione binaria |
| **Logit** | il numero «grezzo» prima della sigmoide; la sigmoide lo trasforma in probabilita' |
| **Epoca** | un passaggio completo su tutti i dati di addestramento |
| **Batch** | il pacchetto di esempi elaborato in un passo di addestramento |
| **Learning rate** | ampiezza del passo di aggiornamento dei pesi |
| **Adam** | ottimizzatore adattivo, lo standard di partenza |
| **MLP** (*multi-layer perceptron*) | la rete neurale di base: strati lineari alternati ad attivazioni |
| **ReLU** | attivazione max(0, x): introduce la non linearita' |
| **Early stopping** | fermarsi quando la validazione smette di migliorare: antidoto all'overfitting |

## 0. Verifica dell'ambiente

Prima di tutto importiamo le librerie e fissiamo il seme. Se una libreria manca, questa cella lo dice subito: meglio scoprirlo ora che a meta' mattinata. (La verifica completa dell'ambiente si fa con il Notebook 0 in apertura del corso.)

In [ ]:
# --- Import di tutte le librerie della mattinata -------------------------
# Ogni import e' commentato con il ruolo che la libreria avra' oggi.

import sys, platform
import numpy as np                      # calcolo numerico: array, generatore casuale
import pandas as pd                     # tabelle (DataFrame): caricamento e manipolazione dati
import matplotlib.pyplot as plt         # grafici: ROC, PR, calibrazione, curve di apprendimento

from scipy import stats                 # test statistici: binomiale, Beta (Jeffreys)

import sklearn                          # scikit-learn: baseline, metriche, preprocessing
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve,
                             average_precision_score, precision_recall_curve,
                             brier_score_loss, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

import xgboost as xgb                   # il nostro champion: gradient boosting

print("Python     :", sys.version.split()[0], "|", platform.system())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost    :", xgb.__version__)

# Seme globale: rende riproducibili campionamenti e addestramenti
SEED = 42
np.random.seed(SEED)
print("\nAmbiente pronto. SEED =", SEED)

---
# Sezione 1 — Il problema PD e perche' l'accuracy inganna

**Richiamo della pillola 1.** Il nostro compito e' stimare la PD: una **classificazione binaria** (default si'/no entro l'orizzonte) il cui output utile e' pero' la **probabilita'**, non la classe. Il fenomeno e' **sbilanciato**: i default sono una piccola minoranza. Ed e' qui che l'accuracy — l'unica metrica oggi in uso — tradisce: su un portafoglio con il 3% di default, il «modello» che risponde sempre *nessun default* raggiunge il **97% di accuracy** restando perfettamente inutile. Lo verifichiamo tra poco, con i numeri.

## 1.1 Caricamento del dataset

Il loader cerca prima il file reale, poi il segnaposto sintetico; in mancanza di entrambi si ferma con le istruzioni.

In [ ]:
# --- Caricamento dati -----------------------------------------------------
import os

PERCORSO_REALE     = "dati/credit_default_taiwan.csv"            # il file UCI fornito col corso
PERCORSO_SEGNAPOSTO = "dati/credit_default_taiwan_sintetico.csv"  # segnaposto per far girare il codice

if os.path.exists(PERCORSO_REALE):
    df = pd.read_csv(PERCORSO_REALE)
    print("Caricato il dataset REALE:", PERCORSO_REALE)
elif os.path.exists(PERCORSO_SEGNAPOSTO):
    df = pd.read_csv(PERCORSO_SEGNAPOSTO)
    print("ATTENZIONE: caricato il SEGNAPOSTO SINTETICO:", PERCORSO_SEGNAPOSTO)
    print("I numeri che seguono servono solo a provare il codice, non a trarre conclusioni.")
else:
    raise FileNotFoundError(
        "Nessun dataset trovato. Posizionare il file del corso in " + PERCORSO_REALE)

# Normalizziamo il nome del target, qualunque sia la fonte
for candidato in ["default payment next month", "default.payment.next.month", "dpnm", "Y"]:
    if candidato in df.columns:
        df = df.rename(columns={candidato: "default"})
if "ID" in df.columns:
    df = df.drop(columns=["ID"])     # l'identificativo non e' un predittore

print("\nDimensioni:", df.shape[0], "righe x", df.shape[1], "colonne")
df.head()

**Le colonne, in breve** (dizionario del dataset UCI):

- `LIMIT_BAL` — fido accordato (dollari taiwanesi); `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` — anagrafica;
- `PAY_0`, `PAY_2` … `PAY_6` — **storico dei ritardi** negli ultimi 6 mesi (−2 = nessun utilizzo, −1/0 = regolare, 1, 2, 3… = mesi di ritardo). Sono le variabili *comportamentali*, le piu' predittive;
- `BILL_AMT1…6` — saldo fatturato in ciascuno dei 6 mesi; `PAY_AMT1…6` — importo pagato in ciascun mese;
- `default` — **il target**: 1 se default il mese successivo.

## 1.2 Quanto e' sbilanciato il fenomeno?

In [ ]:
# --- Tasso di default -----------------------------------------------------
tasso_default = df["default"].mean()
print(f"Clienti totali : {len(df):,}")
print(f"Default (1)    : {df['default'].sum():,}")
print(f"Tasso di default: {tasso_default:.1%}")

# Nel dataset Taiwan il tasso e' intorno al 22%: alto rispetto a un portafoglio
# retail europeo (tipicamente 1-5%). Per l'esempio shock che segue costruiamo
# quindi anche un sotto-campione riportato al 3%, piu' realistico.

## 1.3 L'esempio shock: il modello banale sul portafoglio al 3%

Costruiamo un **sotto-campione al 3% di default** (teniamo tutti i buoni, campioniamo i cattivi) e su di esso valutiamo il *modello banale* che risponde sempre «nessun default» — in scikit-learn e' il `DummyClassifier(strategy="most_frequent")`. Guardiamo accuracy **e** recall.

In [ ]:
# --- Sotto-campione al 3% e modello banale --------------------------------
buoni   = df[df["default"] == 0]
cattivi = df[df["default"] == 1]

# quanti cattivi servono perche' siano il 3% del nuovo campione?
n_cattivi_3pct = int(len(buoni) * 0.03 / 0.97)
cattivi_camp = cattivi.sample(n=n_cattivi_3pct, random_state=SEED)
df_3pct = pd.concat([buoni, cattivi_camp]).sample(frac=1, random_state=SEED)  # mescola

X_3 = df_3pct.drop(columns=["default"])
y_3 = df_3pct["default"]
print(f"Portafoglio simulato: {len(df_3pct):,} clienti, tasso di default {y_3.mean():.1%}")

banale = DummyClassifier(strategy="most_frequent")   # risponde sempre la classe maggioritaria
banale.fit(X_3, y_3)
pred_banale = banale.predict(X_3)

print(f"\nAccuracy del modello banale : {accuracy_score(y_3, pred_banale):.1%}")
print(f"Recall del modello banale   : {recall_score(y_3, pred_banale):.1%}")
print("\nLettura: accuracy altissima, eppure il modello non trova NEMMENO UN default.")
print("Su un fenomeno sbilanciato l'accuracy da sola non dice nulla: ci servono metriche migliori.")

## 1.4 Train / validation / test

D'ora in poi lavoriamo sul dataset **completo** (piu' default = stime piu' stabili) e lo dividiamo in tre parti, **stratificando** (cioe' mantenendo lo stesso tasso di default in ogni parte):

- **train** (60%): per addestrare i modelli;
- **validation** (20%): per le scelte intermedie (iperparametri, early stopping);
- **test** (20%): per il giudizio finale — lo tocchiamo solo alla fine, mai per scegliere.

Separare le tre parti e' la prima regola igienica: valutare un modello sugli stessi dati su cui ha imparato significa premiare l'overfitting.

In [ ]:
# --- Split stratificato 60/20/20 ------------------------------------------
X = df.drop(columns=["default"])
y = df["default"]

# primo taglio: 60% train, 40% resto
X_train, X_resto, y_train, y_resto = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED)
# secondo taglio: il resto a meta' -> 20% validation, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_resto, y_resto, test_size=0.50, stratify=y_resto, random_state=SEED)

for nome, parte in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{nome:11s}: {len(parte):6,} clienti | tasso default {parte.mean():.2%}")

## 1.5 Le prime baseline: regressione logistica e XGBoost (il champion)

Due riferimenti prima di qualunque rete:

- la **regressione logistica**: il modello lineare classico del credit scoring, semplice e interpretabile. Le serve lo **scaling** (riportare le variabili su scale comparabili), quindi la mettiamo in *pipeline* con `StandardScaler`;
- **XGBoost**: il gradient boosting che il team gia' usa in produzione. Da qui in avanti e' il nostro **champion**: ogni modello del corso dovra' confrontarsi con lui, sugli stessi dati e con le stesse metriche. Notare: a XGBoost lo scaling non serve (lavora per soglie, non per distanze).

In [ ]:
# --- Baseline 1: regressione logistica (con scaling in pipeline) ----------
logistica = make_pipeline(
    StandardScaler(),                                  # scaling: media 0, deviazione 1
    LogisticRegression(max_iter=2000, random_state=SEED))
logistica.fit(X_train, y_train)

# predict_proba restituisce due colonne [P(classe 0), P(classe 1)]: teniamo la seconda
proba_log_val  = logistica.predict_proba(X_val)[:, 1]
proba_log_test = logistica.predict_proba(X_test)[:, 1]
print("Logistica addestrata.")

# --- Baseline 2 (champion): XGBoost ---------------------------------------
champion = xgb.XGBClassifier(
    n_estimators=300,        # numero di alberi
    learning_rate=0.1,       # contributo di ciascun albero
    max_depth=4,             # profondita' massima: alberi poco profondi generalizzano meglio
    subsample=0.8,           # frazione di righe per albero (robustezza)
    colsample_bytree=0.8,    # frazione di colonne per albero
    eval_metric="auc",       # metrica monitorata
    random_state=SEED)
champion.fit(X_train, y_train)

proba_xgb_val  = champion.predict_proba(X_val)[:, 1]
proba_xgb_test = champion.predict_proba(X_test)[:, 1]
print("XGBoost (champion) addestrato.")

## 1.6 La cassetta delle metriche di discriminazione

Scriviamo **una volta sola** la funzione che calcola tutte le metriche della mattinata; la riuseremo per ogni modello del corso. Dentro ci sono:

- le metriche **a soglia** (accuracy, precision, recall, F1): dipendono dalla soglia scelta (qui 0,5);
- le metriche **di ordinamento** (AUC → Gini, PR-AUC, KS): non dipendono dalla soglia, guardano se il modello mette i cattivi davanti ai buoni — ed e' questo che interessa a un modello di PD;
- il **Brier score**, che anticipa il tema calibrazione della sezione 2.

In [ ]:
# --- Funzione unica di valutazione ----------------------------------------
def valuta_modello(y_vero, proba, soglia=0.5):
    """Restituisce un dizionario con tutte le metriche della mattinata.

    y_vero : array 0/1 osservato
    proba  : probabilita' di default stimata dal modello
    soglia : soglia per trasformare la probabilita' in classe (default 0,5)
    """
    classe = (np.asarray(proba) >= soglia).astype(int)

    # metriche a soglia
    out = {
        "accuracy":  accuracy_score(y_vero, classe),
        "precision": precision_score(y_vero, classe, zero_division=0),
        "recall":    recall_score(y_vero, classe),
        "F1":        f1_score(y_vero, classe),
    }
    # metriche di ordinamento (indipendenti dalla soglia)
    auc = roc_auc_score(y_vero, proba)
    out["AUC"]    = auc
    out["Gini"]   = 2 * auc - 1                       # la scala dei validatori
    out["PR-AUC"] = average_precision_score(y_vero, proba)
    # KS: massima distanza verticale tra le due curve cumulate = max(TPR - FPR)
    fpr, tpr, _ = roc_curve(y_vero, proba)
    out["KS"] = float(np.max(tpr - fpr))
    # qualita' delle probabilita' (anticipo della sezione 2)
    out["Brier"] = brier_score_loss(y_vero, proba)
    return out

def semaforo_gini(gini):
    """Lettura di prassi del Gini (ipotesi I1 del corso: prassi, non norma)."""
    if gini < 0.40:  return "DEBOLE (< 40%)"
    if gini <= 0.60: return "ACCETTABILE (40-60%)"
    return "BUONO (> 60%)"

print("Funzioni pronte: valuta_modello() e semaforo_gini().")

In [ ]:
# --- Confronto sul TEST: banale vs logistica vs champion -------------------
banale_pieno = DummyClassifier(strategy="prior")  # 'prior': probabilita' = tasso medio
banale_pieno.fit(X_train, y_train)
proba_ban_test = banale_pieno.predict_proba(X_test)[:, 1]

risultati = pd.DataFrame({
    "Banale":     valuta_modello(y_test, proba_ban_test),
    "Logistica":  valuta_modello(y_test, proba_log_test),
    "XGBoost":    valuta_modello(y_test, proba_xgb_test),
}).T.round(4)

print(risultati.to_string())
print()
for nome, proba in [("Logistica", proba_log_test), ("XGBoost", proba_xgb_test)]:
    g = 2 * roc_auc_score(y_test, proba) - 1
    print(f"Gini {nome:10s}: {g:.1%}  ->  {semaforo_gini(g)}")

**Lettura della tabella.** Il modello banale ha accuracy rispettabile e Gini nullo: ancora una volta, l'accuracy non distingue un modello da un sasso. Tra logistica e XGBoost confrontiamo soprattutto **Gini, PR-AUC e KS**: sono le metriche di *ordinamento*, quelle che contano per una PD. La matrice di confusione e le curve, qui sotto, danno il quadro visivo.

In [ ]:
# --- Matrice di confusione del champion (soglia 0,5) -----------------------
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, (proba_xgb_test >= 0.5).astype(int),
    display_labels=["buono (0)", "default (1)"], colorbar=False, ax=ax)
ax.set_title("XGBoost - matrice di confusione (soglia 0,5)")
plt.tight_layout(); plt.show()

# In alto a sinistra i veri negativi, in basso a destra i veri positivi.
# Gli errori: falsi positivi (buoni segnalati) e falsi negativi (default mancati).

In [ ]:
# --- Curve ROC e Precision-Recall ------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

for nome, proba in [("Logistica", proba_log_test), ("XGBoost", proba_xgb_test)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax1.plot(fpr, tpr, label=f"{nome} (AUC {roc_auc_score(y_test, proba):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax2.plot(rec, prec, label=f"{nome} (PR-AUC {average_precision_score(y_test, proba):.3f})")

ax1.plot([0, 1], [0, 1], "k--", lw=1, label="caso (AUC 0,5)")
ax1.set_xlabel("FPR (falsi allarmi)"); ax1.set_ylabel("TPR = recall")
ax1.set_title("Curva ROC"); ax1.legend()

ax2.axhline(y_test.mean(), color="k", ls="--", lw=1, label="caso (= tasso di default)")
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.set_title("Curva Precision-Recall"); ax2.legend()
plt.tight_layout(); plt.show()

# Nota: nella ROC il 'caso' e' la diagonale; nella PR il 'caso' e' il tasso di default.
# Con classi sbilanciate la PR e' piu' severa e piu' informativa della ROC.

In [ ]:
# --- Il KS, visto sulle distribuzioni dei punteggi --------------------------
# Disegniamo le distribuzioni cumulate del punteggio per buoni e cattivi:
# il KS e' la massima distanza verticale tra le due.
punteggio = proba_xgb_test
fpr, tpr, soglie = roc_curve(y_test, punteggio)
soglie = np.clip(soglie, 0, 1)        # la prima soglia e' infinita: la riportiamo in [0, 1]
ks_valore = np.max(tpr - fpr)
ks_indice = int(np.argmax(tpr - fpr))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(soglie, tpr, label="cumulata dei DEFAULT (TPR)")
ax.plot(soglie, fpr, label="cumulata dei BUONI (FPR)")
ax.vlines(soglie[ks_indice], fpr[ks_indice], tpr[ks_indice],
          color="red", lw=2, label=f"KS = {ks_valore:.3f}")
ax.set_xlim(1, 0)   # soglia decrescente da sinistra a destra
ax.set_xlabel("soglia di punteggio"); ax.set_ylabel("quota cumulata")
ax.set_title("XGBoost - statistica KS")
ax.legend(); plt.tight_layout(); plt.show()

---
# Sezione 2 — La PD e' una probabilita': la calibrazione

**Richiamo della pillola 2.** La PD non serve (solo) a classificare: **alimenta pricing, accantonamenti e capitale**. Un modello che ordina benissimo ma stima «3%» dove la frequenza vera e' 6% produce prezzi e accantonamenti sbagliati — e un validatore se ne accorge subito. La proprieta' che ci serve si chiama **calibrazione**: tra tutti i clienti a cui diciamo «PD ≈ x%», la quota di default osservata deve essere ≈ x%.

## 2.1 Reliability diagram e Brier score

In [ ]:
# --- Reliability diagram ----------------------------------------------------
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for nome, proba in [("Logistica", proba_log_test), ("XGBoost", proba_xgb_test)]:
    # calibration_curve raggruppa per fasce di probabilita' stimata
    frac_oss, pd_media = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")
    ax.plot(pd_media, frac_oss, "o-", label=f"{nome} (Brier {brier_score_loss(y_test, proba):.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="calibrazione perfetta")
lim = max(proba_log_test.max(), proba_xgb_test.max()) * 1.05
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("PD media stimata (per fascia)")
ax.set_ylabel("tasso di default osservato")
ax.set_title("Reliability diagram (test)")
ax.legend(); plt.tight_layout(); plt.show()

# Punti SULLA diagonale = ben calibrato. Sotto la diagonale = PD sovrastimate;
# sopra = PD sottostimate (il caso che preoccupa la Vigilanza).

## 2.2 Le fasce di rating e i test per fascia

I validatori non guardano la calibrazione «in media»: la guardano **per fascia di rating**. Costruiamo le fasce come fa la pratica: ordiniamo i clienti per PD stimata e li dividiamo in gruppi (qui 8 fasce per quantili, dalla meno alla piu' rischiosa). Per ogni fascia calcoliamo PD media stimata, default osservati, e due test:

- **test binomiale (esatto)**: se la PD stimata fosse quella vera, osservare questi default sarebbe plausibile? p-value piccolo = no;
- **test di Jeffreys** (istruzioni BCE per il *validation reporting*): variante bayesiana — con i default osservati, la distribuzione a posteriori della PD vera e' una Beta(d + ½, n − d + ½); il p-value e' la probabilita' che la PD vera sia ≤ di quella stimata. **p-value piccolo = la PD stimata sottostima** il rischio della fascia.

In [ ]:
# --- Fasce di rating sul champion e test di calibrazione --------------------
def tabella_calibrazione(y_vero, proba, n_fasce=8):
    """Tabella per fascia: n, default, PD stimata, tasso osservato, p-value binomiale e Jeffreys."""
    tab = pd.DataFrame({"proba": np.asarray(proba), "y": np.asarray(y_vero)})
    # qcut: fasce per quantili della PD stimata (1 = meno rischiosa)
    tab["fascia"] = pd.qcut(tab["proba"], q=n_fasce, labels=False, duplicates="drop") + 1

    righe = []
    for f, g in tab.groupby("fascia"):
        n, d = len(g), int(g["y"].sum())
        pd_stimata, tasso_oss = g["proba"].mean(), g["y"].mean()
        # binomiale esatto, bilaterale
        p_binom = stats.binomtest(d, n, pd_stimata).pvalue
        # Jeffreys: P(PD_vera <= PD_stimata | dati) con posteriori Beta(d+1/2, n-d+1/2)
        p_jeff = stats.beta.cdf(pd_stimata, d + 0.5, n - d + 0.5)
        righe.append({"fascia": int(f), "n": n, "default": d,
                      "PD stimata": pd_stimata, "tasso osservato": tasso_oss,
                      "p binomiale": p_binom, "p Jeffreys": p_jeff})
    return pd.DataFrame(righe).set_index("fascia")

tab_cal = tabella_calibrazione(y_test, proba_xgb_test, n_fasce=8)

# esito 'da validatore': Jeffreys sotto il 5% -> sospetta sottostima nella fascia
tab_cal["esito"] = np.where(tab_cal["p Jeffreys"] < 0.05, "SOTTOSTIMA?", "ok")
print(tab_cal.round(4).to_string())
print("""
Come la legge un validatore: fascia per fascia, la PD stimata regge il confronto
con i default osservati? Un p-value di Jeffreys molto piccolo segnala che la
fascia sottostima il rischio; molti 'ok' = calibrazione difendibile.""")

---
# Sezione 3 — PyTorch essenziale e il primo MLP

**Richiamo della pillola 3.** Fin qui il mondo scikit-learn: `fit` e `predict`, l'addestramento e' una scatola chiusa. In PyTorch la scatola si apre: scriviamo noi il ciclo di addestramento. Sembra un passo indietro, ma e' il motivo per cui PyTorch e' lo standard della ricerca: controllo totale. E vedere il ciclo scritto per esteso e' il modo migliore per capire *che cosa significa* addestrare una rete.

Il percorso: **tensori** → **autograd** → **nn.Module** → **training loop** → primo **MLP** sul nostro dataset.

## 3.1 I tensori: i mattoni

In [ ]:
# --- Tensori: creazione e operazioni ----------------------------------------
import torch
import torch.nn as nn

torch.manual_seed(SEED)   # riproducibilita' anche per PyTorch
print("PyTorch", torch.__version__, "| GPU disponibile:", torch.cuda.is_available(),
      "(oggi lavoriamo volutamente su CPU)")

# Un tensore e' l'array multidimensionale di PyTorch: il cugino del ndarray NumPy
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("\na =\n", a)
print("forma:", tuple(a.shape), "| tipo:", a.dtype)

# Operazioni come in NumPy
print("\na * 10 =\n", a * 10)
print("\nmedia per colonna:", a.mean(dim=0))

# Andata e ritorno con NumPy (condividono la memoria: attenzione alle modifiche)
b = torch.from_numpy(np.array([1.0, 2.0, 3.0]))
print("\nda NumPy:", b, "-> di nuovo a NumPy:", b.numpy())

## 3.2 Autograd: le derivate calcolate per noi

L'addestramento di una rete e' una discesa lungo il **gradiente** della loss. Il miracolo di PyTorch e' **autograd**: dichiariamo `requires_grad=True` su un tensore, componiamo calcoli qualsiasi, chiamiamo `.backward()` — e PyTorch calcola da solo tutte le derivate, all'indietro lungo il grafo dei calcoli. Verifichiamolo su una funzione di cui conosciamo la derivata a mano.

In [ ]:
# --- Autograd su un esempio controllabile a mano ----------------------------
# f(x) = x^2 + 3x   ->   f'(x) = 2x + 3   ->   in x = 2: f'(2) = 7
x = torch.tensor(2.0, requires_grad=True)   # chiediamo di tracciare i gradienti
f = x**2 + 3*x                              # componiamo il calcolo
f.backward()                                # derivate all'indietro

print("f(2)  =", f.item())        # .item(): dal tensore al numero Python
print("f'(2) =", x.grad.item(), " (atteso: 7.0)")

# Questo e' TUTTO il meccanismo: nella rete, al posto di x ci saranno migliaia
# di pesi, e al posto di f la loss. backward() non cambia di una virgola.

## 3.3 Preparazione dei dati per la rete

Due differenze pratiche rispetto a XGBoost, da fissare subito:

1. **lo scaling e' necessario**: le reti apprendono per gradienti, e variabili su scale molto diverse (eta' ~40, fido ~200.000) rendono la discesa instabile. Agli alberi non serviva: lavorano per soglie. *(Questa differenza tornera', approfondita, nella mattina 2.)*
2. i dati diventano **tensori `float32`**, impacchettati da un `DataLoader` che li serve alla rete in **batch** mescolati.

Riusiamo lo scaler gia' addestrato sul train (mai ri-adattarlo su validation o test: sarebbe una fuga di informazione).

In [ ]:
# --- Da DataFrame a tensori e DataLoader ------------------------------------
from torch.utils.data import TensorDataset, DataLoader

scaler = StandardScaler().fit(X_train)            # adattato SOLO sul train

def in_tensori(X_df, y_serie):
    """Applica lo scaling e converte in tensori float32 (y come colonna)."""
    X_t = torch.tensor(scaler.transform(X_df), dtype=torch.float32)
    y_t = torch.tensor(y_serie.to_numpy(), dtype=torch.float32).reshape(-1, 1)
    return X_t, y_t

Xtr_t, ytr_t = in_tensori(X_train, y_train)
Xva_t, yva_t = in_tensori(X_val,   y_val)
Xte_t, yte_t = in_tensori(X_test,  y_test)

# Dataset + DataLoader: batch da 256 esempi, mescolati a ogni epoca
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=256, shuffle=True)

print("Tensori pronti:", tuple(Xtr_t.shape), "train |", tuple(Xva_t.shape), "val |",
      tuple(Xte_t.shape), "test")

## 3.4 Il primo MLP, dichiarato con `nn.Module`

Un **MLP** (*multi-layer perceptron*) e' una pila di strati lineari alternati ad attivazioni non lineari (**ReLU**). Il nostro: ingresso → 64 neuroni → 32 neuroni → 1 uscita. L'uscita e' un **logit** (numero grezzo): la sigmoide che lo trasforma in probabilita' e' *dentro* la loss `BCEWithLogitsLoss`, scelta piu' stabile numericamente del fare sigmoide + BCE a mano.

In `nn.Module` si scrivono due cose: i **pezzi** (nel costruttore) e il **flusso** (`forward`).

In [ ]:
# --- Definizione della rete --------------------------------------------------
class RetePD(nn.Module):
    """MLP minimo per la stima della PD: ingresso -> 64 -> 32 -> 1 logit."""

    def __init__(self, n_ingressi):
        super().__init__()                      # inizializza la macchina nn.Module
        self.strati = nn.Sequential(
            nn.Linear(n_ingressi, 64),          # strato lineare: n_ingressi -> 64
            nn.ReLU(),                          # non linearita'
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),                   # uscita: UN logit (non una probabilita')
        )

    def forward(self, x):
        """Il flusso in avanti: come l'input attraversa la rete."""
        return self.strati(x)

torch.manual_seed(SEED)                         # pesi iniziali riproducibili
rete = RetePD(n_ingressi=Xtr_t.shape[1])
print(rete)
n_parametri = sum(p.numel() for p in rete.parameters())
print(f"\nParametri da apprendere: {n_parametri:,}")

## 3.5 Il training loop, riga per riga

Ecco il cuore di PyTorch. Prima, per confronto, **come lo faremmo in scikit-learn** (il mondo da cui veniamo):

```python
modello.fit(X_train, y_train)          # tutto qui: il ciclo e' nascosto
```

E ora **la stessa cosa scritta per esteso** in PyTorch — cinque mosse che si ripetono per ogni batch, per ogni epoca:

1. `forward`: la rete produce i logit del batch;
2. `loss`: confrontiamo i logit con le etichette vere (BCE);
3. `zero_grad`: azzeriamo i gradienti del giro precedente;
4. `backward`: autograd calcola i gradienti della loss rispetto a ogni peso;
5. `step`: l'ottimizzatore (**Adam**) aggiorna i pesi di un passetto (il **learning rate**).

A fine epoca misuriamo l'**AUC sulla validation** e teniamo da parte i pesi migliori (**early stopping** in versione semplice).

In [ ]:
# --- Addestramento -----------------------------------------------------------
import copy

loss_bce  = nn.BCEWithLogitsLoss()                      # sigmoide + BCE, in un colpo solo
ottimizza = torch.optim.Adam(rete.parameters(), lr=1e-3) # Adam con learning rate 0,001

N_EPOCHE = 25
storia = {"loss_train": [], "auc_val": []}
migliore = {"auc": -1.0, "pesi": None, "epoca": -1}

for epoca in range(1, N_EPOCHE + 1):
    # ---- fase di addestramento ----
    rete.train()                                        # modalita' addestramento
    loss_epoca = 0.0
    for X_batch, y_batch in train_loader:
        logit = rete(X_batch)                           # 1) forward
        loss  = loss_bce(logit, y_batch)                # 2) loss sul batch
        ottimizza.zero_grad()                           # 3) azzera i gradienti
        loss.backward()                                 # 4) gradienti via autograd
        ottimizza.step()                                # 5) aggiorna i pesi
        loss_epoca += loss.item() * len(X_batch)
    loss_epoca /= len(Xtr_t)

    # ---- valutazione su validation (niente gradienti: solo lettura) ----
    rete.eval()                                         # modalita' valutazione
    with torch.no_grad():
        proba_val = torch.sigmoid(rete(Xva_t)).numpy().ravel()
    auc_val = roc_auc_score(y_val, proba_val)

    storia["loss_train"].append(loss_epoca)
    storia["auc_val"].append(auc_val)
    if auc_val > migliore["auc"]:                       # early stopping 'semplice':
        migliore = {"auc": auc_val,                     # memorizziamo i pesi migliori
                    "pesi": copy.deepcopy(rete.state_dict()),
                    "epoca": epoca}

    if epoca % 5 == 0 or epoca == 1:
        print(f"epoca {epoca:2d} | loss train {loss_epoca:.4f} | AUC val {auc_val:.4f}")

rete.load_state_dict(migliore["pesi"])                  # ripristiniamo i pesi migliori
print(f"\nPesi migliori: epoca {migliore['epoca']} (AUC validation {migliore['auc']:.4f})")

In [ ]:
# --- Curve di apprendimento --------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
epoche = range(1, len(storia["loss_train"]) + 1)
ax1.plot(epoche, storia["loss_train"]); ax1.set_title("Loss di addestramento")
ax1.set_xlabel("epoca"); ax1.set_ylabel("BCE")
ax2.plot(epoche, storia["auc_val"]); ax2.set_title("AUC su validation")
ax2.axvline(migliore["epoca"], color="red", ls="--", lw=1, label=f"migliore: epoca {migliore['epoca']}")
ax2.set_xlabel("epoca"); ax2.set_ylabel("AUC"); ax2.legend()
plt.tight_layout(); plt.show()

# La loss scende sempre (la rete memorizza); l'AUC di validation prima sale poi
# si appiattisce o peggiora: QUELLO e' il momento di fermarsi (early stopping).

## 3.6 Il verdetto della mattina: confronto a tre sul test

Il momento champion/challenger: valutiamo l'MLP sul **test** (mai toccato finora) con le **stesse metriche** di logistica e XGBoost. La tabella che ne esce e' il primo esempio del formato che chiudera' ogni laboratorio del corso.

In [ ]:
# --- Tabella finale champion/challenger --------------------------------------
rete.eval()
with torch.no_grad():
    proba_mlp_test = torch.sigmoid(rete(Xte_t)).numpy().ravel()

finale = pd.DataFrame({
    "Logistica":        valuta_modello(y_test, proba_log_test),
    "XGBoost (champion)": valuta_modello(y_test, proba_xgb_test),
    "MLP (challenger)": valuta_modello(y_test, proba_mlp_test),
}).T.round(4)

print(finale.to_string())
print()
for nome, proba in [("Logistica", proba_log_test),
                    ("XGBoost  ", proba_xgb_test),
                    ("MLP      ", proba_mlp_test)]:
    g = 2 * roc_auc_score(y_test, proba) - 1
    print(f"Gini {nome}: {g:.1%}  ->  {semaforo_gini(g)}")

**Come leggere il verdetto.**

- Sul **tabellare medio** come questo, e' del tutto normale che il champion XGBoost resti davanti o alla pari: la letteratura di benchmark (Grinsztajn 2022; Shwartz-Ziv & Armon 2022; Borisov 2024) dice esattamente questo. Il challenger di oggi e' un MLP *minimo*, senza nessuno degli accorgimenti della mattina 2 (embedding, regolarizzazione, gestione dello sbilanciamento, calibrazione);
- il punto della mattina non era vincere: era **misurare con le metriche giuste** — Gini, PR-AUC, KS, Brier — e avere un ciclo PyTorch funzionante tra le mani;
- domani (mattina 2) equipaggiamo il challenger sul serio; dopodomani (mattina 3) lo portiamo dove XGBoost non puo' seguirlo: le **sequenze di transazioni**.

---
# Sintesi della mattina

1. L'accuracy da sola **non valuta** un modello di PD: servono metriche di **discriminazione** (Gini, PR-AUC, KS) e di **calibrazione** (reliability, Brier, test binomiale e di Jeffreys per fascia);
2. abbiamo una **funzione unica di valutazione** e un **champion** (XGBoost): ogni modello del corso passera' da li';
3. sappiamo leggere una **tabella di calibrazione per fasce** come la legge un validatore;
4. abbiamo scritto **per esteso** l'addestramento di una rete: tensori → autograd → nn.Module → training loop → MLP.

---
# Esercizi — a difficolta' crescente

Celle predisposte qui sotto; le **soluzioni** arrivano a fine corso. Gli esercizi 4 e 5 sono opzionali, per chi vuole spingersi oltre.

**E1 (guidato) — La soglia non e' un dogma.** Con `valuta_modello(y_test, proba_xgb_test, soglia=s)` calcolare precision e recall del champion per `s` in {0,2, 0,3, 0,5, 0,7}. Che cosa succede alzando la soglia? Perche' per un modello di PD la soglia conta meno che per un sistema di allerta?

**E2 (guidato) — Il KS a mano.** Ricalcolare la statistica KS *senza* usare `roc_curve`: ordinare i punteggi, costruire le due cumulate (buoni e cattivi) e prenderne la massima distanza. Confrontare con il valore della sezione 1.

**E3 — Quante fasce?** Rifare la tabella di calibrazione con 6 e con 10 fasce. Come cambiano i p-value? Perche' con piu' fasce i test diventano meno potenti? (Suggerimento: numerosita' per fascia.)

**E4 (opzionale) — L'architettura conta?** Modificare `RetePD`: provare 128→64 e un terzo strato. L'AUC di validation migliora? E il divario con XGBoost? Attenzione ai segnali di overfitting nelle curve di apprendimento.

**E5 (opzionale) — Il campione al 3%.** Riaddestrare logistica e XGBoost sul sotto-campione `df_3pct` (train/val/test stratificati). Confrontare Gini e PR-AUC con quelli sul dataset pieno: quale metrica soffre di piu' lo sbilanciamento spinto?

In [ ]:
# === E1 - la soglia ==========================================================
# TODO: ciclo sulle soglie {0.2, 0.3, 0.5, 0.7} e stampa di precision e recall


In [ ]:
# === E2 - KS a mano ==========================================================
# TODO: cumulate di buoni e cattivi sui punteggi ordinati, massima distanza


In [ ]:
# === E3 - numero di fasce ====================================================
# TODO: tabella_calibrazione con n_fasce=6 e n_fasce=10, confronto p-value


In [ ]:
# === E4 (opzionale) - architettura ===========================================
# TODO: nuova classe o parametrizzazione di RetePD, riaddestrare, confrontare


In [ ]:
# === E5 (opzionale) - campione al 3% =========================================
# TODO: split su df_3pct, riaddestrare logistica e XGBoost, confrontare le metriche


---
*Riferimenti.* Dataset: Yeh, I-C., Lien, C-H. (2009), «The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients», *Expert Systems with Applications*, 36(2) — UCI Machine Learning Repository, id 350. Le soglie e i test citati seguono le ipotesi di lavoro del corso (quadro di riferimento, I1–I3), dichiarate come prassi e in attesa di validazione da parte della funzione di validazione interna.

*Prossimo appuntamento: Mattina 2 — Reti su dati tabellari, fatte bene.*